In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
import os
groq_api_key = os.getenv("GROQ_API_KEY") or None


In [6]:
from langchain_groq.chat_models import ChatGroq
model=ChatGroq(api_key=groq_api_key,model="openai/gpt-oss-120b") # type: ignore

c:\Users\user\Desktop\Hr\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 281.27it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### RAG MODEL

In [8]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings


In [9]:
class Embeddings(Embeddings):
    def __init__(self, embedder: SentenceTransformer):
        self.embedder = embedder

    def embed_documents(self, texts):
        return self.embedder.encode(texts).tolist()

    def embed_query(self, text):
        return self.embedder.encode(text).tolist()
    
    def __call__(self, text):
        return self.embed_query(text)

In [10]:
class Rag:
    def __init__(self,model_name:ChatGroq,embedder:SentenceTransformer): 
        self.model=model_name
        self.embedding=Embeddings(embedder)
        self.text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=50)
        self.vector_store=None
        self.path=None
    
    def create_vector_store(self,path:str):
        loader = PyMuPDFLoader(path)
        documents = loader.load()
        docs = self.text_splitter.split_documents(documents)
        self.vector_store = FAISS.from_documents(docs, self.embedding) #type:ignore
        return self.vector_store

    
    def retrieve(self,query:str):
        if self.vector_store is None:
            raise ValueError("Vector store not created.")
        relevant_docs = self.vector_store.similarity_search(query, k=10)  # Example: retrieve top 5 relevant documents
        return relevant_docs
    
    


    
    


In [12]:
path=r"C:\Users\user\Desktop\Hr\resume (22).pdf"

obj=Rag(model,embedder)

vector_store=obj.create_vector_store(path)

In [19]:
retrved_docs=obj.retrieve("What are the reciepe for making tea?")

In [20]:
context = "\n\n".join([doc.page_content for doc in retrved_docs]).strip()

### Now making resume extractor 

In [50]:
from pydantic import BaseModel
from typing import List,Optional
class Output(BaseModel):
    skills: Optional[List[str]]
    total_experience_years: Optional[int]
    job_titles: Optional[List[str]]
    responsibilities: Optional[List[str]]
    previous_companies: Optional[List[str]]
    education: Optional[List[str]]
    PORS: Optional[List[str]]

In [51]:
llm=model.with_structured_output(Output)

In [52]:
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate,HumanMessagePromptTemplate

template = """You are a helpful assistant for extracting information from resumes.
Given the following context, extract the following information:"""

system_message = SystemMessagePromptTemplate.from_template(template)

human_message = HumanMessagePromptTemplate.from_template("{context}")

In [53]:
final_prompt = ChatPromptTemplate.from_messages([system_message, human_message])

In [54]:
chain=final_prompt | llm

In [55]:
path=r"C:\Users\user\Desktop\Hr\Sample_Backend_Developer_Resume.pdf"
from langchain_community.document_loaders import PyMuPDFLoader
loader = PyMuPDFLoader(path)
documents = loader.load()


In [56]:
context="\n\n".join([doc.page_content for doc in documents])

In [57]:
response=chain.invoke({"context":context})

In [59]:
response.dict()

C:\Users\user\AppData\Local\Temp\ipykernel_3324\3202056457.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  response.dict()


{'skills': ['Python',
  'Django',
  'Django REST Framework (DRF)',
  'REST APIs',
  'PostgreSQL',
  'MySQL',
  'Git',
  'Docker',
  'Redis',
  'AWS (EC2, S3)',
  'PyTest'],
 'total_experience_years': 3,
 'job_titles': ['Backend Developer', 'Junior Python Developer'],
 'responsibilities': ['Developed scalable REST APIs using Django REST Framework',
  'Integrated Redis caching to improve API performance',
  'Deployed applications on AWS EC2',
  'Optimized PostgreSQL database queries',
  'Used Docker for containerization',
  'Built backend modules in Django',
  'Implemented authentication and authorization systems',
  'Wrote unit tests using PyTest',
  'Used Git for version control'],
 'previous_companies': ['CodeCraft Technologies', 'SoftByte Solutions'],
 'education': ['B.Tech in Computer Science – XYZ Institute of Technology (2016 – 2020)'],
 'PORS': ['Technical Lead – College Coding Club', 'Organized 2 Hackathons']}

Bhai mene resume extract krliya h like important parts usme 

In [61]:
def normalize(text):
    return text.lower().strip()

resume_skills = [normalize(s) for s in response.skills]

### Job Description Extractor

In [66]:
class JDOutput(BaseModel):
    required_skills: Optional[List[str]]
    preferred_skills: Optional[List[str]]
    min_experience_years: Optional[int]
    responsibilities: Optional[List[str]]
    education_required: Optional[List[str]]

In [67]:
jd="""Backend Developer (Python/Django)
🏢 Company: TechNova Solutions
📍 Location: Remote / Bangalore
💼 Experience: 2–4 Years
🔹 About the Role

We are looking for a Backend Developer with strong experience in Python and Django to build scalable web applications and REST APIs. The ideal candidate should have hands-on experience in database design, API development, and cloud deployment.

🔹 Required Skills

Python

Django

Django REST Framework (DRF)

REST APIs

PostgreSQL

Git

Basic understanding of Docker

🔹 Preferred Skills (Good to Have)

AWS (EC2, S3)

Redis

Celery

CI/CD pipelines

Unit Testing (PyTest)

🔹 Experience Required

Minimum 2 years of professional backend development experience

Experience building scalable APIs

🔹 Education

B.Tech / B.E. in Computer Science or related field
(or equivalent practical experience)

🔹 Responsibilities

Develop and maintain backend services

Design RESTful APIs

Optimize database queries

Collaborate with frontend team

Deploy applications on cloud infrastructure"""

In [68]:
job_extractor=model.with_structured_output(JDOutput)

template="""You are a helpful assistant for extracting information from job descriptions.
Given the following job description, extract the following information:"""

system_message = SystemMessagePromptTemplate.from_template(template)

human_message = HumanMessagePromptTemplate.from_template("{job_description}")

final_prompt = ChatPromptTemplate.from_messages([system_message, human_message])

In [72]:
chain=final_prompt | job_extractor


jd_response=chain.invoke({"job_description":jd})

In [75]:
dict(jd_response)

{'required_skills': ['Python',
  'Django',
  'Django REST Framework (DRF)',
  'REST APIs',
  'PostgreSQL',
  'Git',
  'Basic understanding of Docker'],
 'preferred_skills': ['AWS (EC2, S3)',
  'Redis',
  'Celery',
  'CI/CD pipelines',
  'Unit Testing (PyTest)'],
 'min_experience_years': 2,
 'responsibilities': ['Develop and maintain backend services',
  'Design RESTful APIs',
  'Optimize database queries',
  'Collaborate with frontend team',
  'Deploy applications on cloud infrastructure'],
 'education_required': ['B.Tech / B.E. in Computer Science or related field (or equivalent practical experience)']}

In [78]:
jd_skills_required=[skill.lower().strip() for skill in jd_response.required_skills]
jd_skills_preffered=[skill.lower().strip() for skill in jd_response.preferred_skills]

### JD skills which are  required and preferred  extracted now comparing resume skills with both type of  skills 

In [101]:
class SkillNormalize(BaseModel):
    normalized_skills: List[str]

In [102]:
classifier_model=ChatGroq(api_key=groq_api_key,model="llama-3.3-70b-versatile")

In [123]:
template="""You are an expert technical recruiter.

Normalize the following skills into their base canonical form. and  remove the  similar skills.

Rules:
- Remove words like: basic, learning, beginner, advanced.
- Remove brackets content.
- Convert abbreviations to standard names (DRF → Django REST Framework).
- Normalize each skill strictly into ONE base canonical skill.
Do not split one skill into multiple skills.
Do not infer additional skills.
Return only cleaned versions of the given skills."""
system_message = SystemMessagePromptTemplate.from_template(template)
human_message = HumanMessagePromptTemplate.from_template("{skills}")
final_prompt = ChatPromptTemplate.from_messages([system_message, human_message])

In [ ]:
skill_extractor=final_prompt | classifier_model.with_structured_output( SkillNormalize)


In [128]:
jd_skills_preffered=[skill.lower().strip() for skill in skill_extractor.invoke({"skills":",".join(jd_skills_preffered)}).normalized_skills] #type: ignore
jd_skills_required=[skill.lower().strip() for skill in skill_extractor.invoke({"skills":",".join(jd_skills_required)}).normalized_skills] #type: ignore
resume_skills=[skill.lower().strip() for skill in skill_extractor.invoke({"skills":",".join(resume_skills)}).normalized_skills] #type: ignore

In [130]:
required_matches = set(jd_skills_required).intersection(set(resume_skills))
preferred_matches = set(jd_skills_preffered).intersection(set(resume_skills))

In [133]:
required_weight = 2
preferred_weight = 1

achieved_score = (
    len(required_matches) * required_weight +
    len(preferred_matches) * preferred_weight
)

max_score = (
    len(jd_skills_required) * required_weight +
    len(jd_skills_preffered) * preferred_weight
)

skills_score = achieved_score / max(max_score, 1)

### Skill Score Calculated Now heading to InterviewProcess